In [ ]:
import torch
from torch import nn
import asyncio
import websockets
import json
import numpy as np

In [ ]:
# 1. Matches your RPSModel class exactly
class RPSModel(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.linear_layer_stack = nn.Sequential(
            nn.Linear(in_features=input_features, out_features=32),
            nn.ReLU(),
            nn.Dropout(p=0.2), # Ensure this matches your DROPOUT variable
            nn.Linear(in_features=32, out_features=16),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(in_features=16, out_features=3)
        )

    def forward(self, x):
        return self.linear_layer_stack(x)

In [ ]:
# 2. Setup constants
INPUT_FEATURES = 48  # The 49 values minus the 1 timestamp
MODEL_PATH = "model_4_weights.pth"
class_names = ["Rock", "Paper", "Scissors"]
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 3. Load Model
model_live = RPSModel(input_features=INPUT_FEATURES, output_features=3)
model_live.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model_live.to(device)
model_live.eval()

print(f"Model loaded on {device}. Ready for {INPUT_FEATURES} features.")

In [ ]:
async def predict_from_websocket(uri):
    print(f"Connecting to {uri}...")
    
    async with websockets.connect(uri) as websocket:
        while True:
            try:
                # Receive raw string
                message = await websocket.recv()
                
                # Split by comma
                parts = message.strip().split(',')
                
                # Validation: Skip header messages or incomplete packets
                if len(parts) != 49:
                    continue
                
                # DROPPING TIMESTAMP: slice from index 1 to the end
                # Convert strings to floats
                features = [float(val) for val in parts[1:]]
                
                # Convert to tensor and add batch dimension: (1, 48)
                input_tensor = torch.tensor(features).float().to(device).unsqueeze(0)
                
                # Inference
                with torch.inference_mode():
                    logits = model_live(input_tensor)
                    prediction = torch.softmax(logits, dim=1)
                    conf, class_idx = torch.max(prediction, dim=1)
                
                # Print result on a single updating line
                label = class_names[class_idx.item()]
                print(f"Prediction: {label} | Confidence: {conf.item():.2%}", end="\r")
                
            except ValueError:
                # In case the websocket sends a non-numeric string unexpectedly
                continue
            except websockets.ConnectionClosed:
                print("\nConnection lost.")
                break
            except KeyboardInterrupt:
                print("\nStopping...")
                break

# To run the live loop:
# await predict_from_websocket("ws://your_imu_server_ip:port")